# Étape 1 — Installation et Gestion de Données

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from category_encoders import CountEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

data_dir = Path("data")

# Load
X_train = pd.read_csv(data_dir / "xtrain_input_Z61KlZo.csv")
y_train = pd.read_csv(data_dir / "ytrain_output_DzPxaPY.csv")
X_test  = pd.read_csv(data_dir / "test_input_5qJzHrr.csv")
print("Données chargées avec succès.")
print(X_train.shape, y_train.shape, X_test.shape)

# Drop cols >=99% NaN (comme toi)
def missing_percentage(df):
    missing_count = df.isna().sum()
    missing_pct = 100 * missing_count / len(df)
    return pd.DataFrame({"nb_missing": missing_count, "pct_missing": missing_pct}).sort_values("pct_missing", ascending=False)

missing_X_train = missing_percentage(X_train)
cols_to_drop_99 = missing_X_train[missing_X_train["pct_missing"] >= 99].index.tolist()
print("Drop >=99% NaN:", cols_to_drop_99)

X_train_clean = X_train.drop(columns=cols_to_drop_99)
X_test_clean  = X_test.drop(columns=cols_to_drop_99)
print(X_train_clean.shape, X_test_clean.shape)

# Types
num_cols = X_train_clean.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train_clean.columns if c not in num_cols]
print(f"Colonnes numériques : {len(num_cols)}")
print(f"Colonnes catégorielles : {len(cat_cols)}")


# Impute num (medianne) + cat ("MISSING")
for col in num_cols:
    med = X_train_clean[col].median()
    X_train_clean[col] = X_train_clean[col].fillna(med)
    X_test_clean[col]  = X_test_clean[col].fillna(med)

for col in cat_cols:
    X_train_clean[col] = X_train_clean[col].fillna("MISSING")
    X_test_clean[col]  = X_test_clean[col].fillna("MISSING")

# Vérification
print("NaN restants dans X_train :", X_train_clean.isna().sum().sum())
print("NaN restants dans X_test  :", X_test_clean.isna().sum().sum())

# Exclude non-explanatory
X_train_model = X_train_clean.drop(columns=["ID"])
X_test_model  = X_test_clean.drop(columns=["ID"])

# Encode categorical
num_cols_model = X_train_model.select_dtypes(include=["number"]).columns.tolist()
cat_cols_model = [c for c in X_train_model.columns if c not in num_cols_model]

encoder = CountEncoder(cols=cat_cols_model)
encoder.fit(X_train_model)

X_train_enc = encoder.transform(X_train_model)
X_test_enc  = encoder.transform(X_test_model)
print(X_train_enc.shape, X_test_enc.shape)

# Aucun NaN ?
print("NaN X_train_enc :", X_train_enc.isna().sum().sum())
print("NaN X_test_enc  :", X_test_enc.isna().sum().sum())
assert X_train_enc.shape[1] == X_test_enc.shape[1]


# Etape 2 -  Cible + split interne

In [ ]:
# Construction de la cible Tweedie (pure premium)
annee = y_train["ANNEE_ASSURANCE"].values
charge = y_train["CHARGE"].values

# Sécurité
pure_premium = np.clip(charge / annee, 0, None)

# Split interne
idx = np.arange(len(X_train_enc))

X_tr, X_val, idx_tr, idx_val, y_tr_pp, y_val_pp = train_test_split(
    X_train_enc,
    idx,
    pure_premium,
    test_size=0.2,
    random_state=42
)

annee_tr = y_train.iloc[idx_tr]["ANNEE_ASSURANCE"].values
annee_val = y_train.iloc[idx_val]["ANNEE_ASSURANCE"].values
charge_val = y_train.iloc[idx_val]["CHARGE"].values

print("Shapes :", X_tr.shape, X_val.shape, y_tr_pp.shape, y_val_pp.shape, idx_tr.shape, idx_val.shape )


# Etape 3 - Entrainement et Optimisation des hyperparametres

In [ ]:
# XGBoost Tweedie
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=190,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=60,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=20,
    objective="reg:tweedie",
    tweedie_variance_power=1.2,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_tr, y_tr_pp)

pp_val_xgb = np.clip(xgb.predict(X_val), 0, None)


In [ ]:
# Prédiction
pp_val_pred = xgb.predict(X_val)
pp_val_pred = np.clip(pp_val_pred, 0, None)

charge_val_pred = pp_val_pred * annee_val

# RMSE
rmse_val = np.sqrt(mean_squared_error(charge_val, charge_val_pred))
print("RMSE CHARGE (XGB Tweedie) =", rmse_val)


In [ ]:
# GLM TWEEDIE
from sklearn.linear_model import TweedieRegressor

glm = TweedieRegressor(
    power=1.2,
    alpha=0.001,   
    link="log",
    max_iter=1000
)

glm.fit(X_tr, y_tr_pp)

pp_val_glm = np.clip(glm.predict(X_val), 0, None)


In [ ]:
# Blend
from sklearn.metrics import mean_squared_error
import numpy as np

for alpha in [0.6, 0.7, 0.8]:
    pp_blend = alpha * pp_val_xgb + (1 - alpha) * pp_val_glm
    charge_blend = pp_blend * annee_val

    rmse = np.sqrt(mean_squared_error(charge_val, charge_blend))
    print(f"alpha={alpha} → RMSE={rmse}")


In [ ]:
# Optimisation de alpha GLm
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

alphas_glm = [1e-4, 3e-4, 1e-3, 3e-3]
best = {"rmse": np.inf}

for a_glm in alphas_glm:
    glm = TweedieRegressor(power=1.2, alpha=a_glm, link="log", max_iter=2000)
    glm.fit(X_tr, y_tr_pp)

    pp_val_glm = np.clip(glm.predict(X_val), 0, None)

    # on prend ton meilleur alpha de blend autour de 0.8, mais on teste finement
    for w in [0.75, 0.80, 0.85, 0.90]:
        pp_blend = w * pp_val_xgb + (1 - w) * pp_val_glm
        charge_blend = pp_blend * annee_val
        rmse = np.sqrt(mean_squared_error(charge_val, charge_blend))

        if rmse < best["rmse"]:
            best = {"rmse": rmse, "alpha_glm": a_glm, "w": w}

    print(f"alpha_glm={a_glm} terminé")

print("BEST:", best)


In [ ]:
# Tester power du GLM
powers = [1.1, 1.2, 1.3]
alphas_glm = [1e-4, 3e-4, 1e-3]

best = {"rmse": np.inf}

for p in powers:
    for a_glm in alphas_glm:
        glm = TweedieRegressor(power=p, alpha=a_glm, link="log", max_iter=2000)
        glm.fit(X_tr, y_tr_pp)

        pp_val_glm = np.clip(glm.predict(X_val), 0, None)

        for w in [0.75, 0.80, 0.85, 0.90]:
            pp_blend = w * pp_val_xgb + (1 - w) * pp_val_glm
            rmse = np.sqrt(mean_squared_error(charge_val, pp_blend * annee_val))

            if rmse < best["rmse"]:
                best = {"rmse": rmse, "power": p, "alpha_glm": a_glm, "w": w}

print("BEST:", best)


---

# Etape 5 - Réentraînement + Prédiction sur X_test + submission.csv

In [ ]:
xgb_final = XGBRegressor(
    n_estimators=190,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=60,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=20,
    objective="reg:tweedie",
    tweedie_variance_power=1.2,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
xgb_final.fit(X_train_enc, pure_premium)

glm_final = TweedieRegressor(
    power=1.1,
    alpha=1e-4,
    link="log",
    max_iter=3000
)
glm_final.fit(X_train_enc, pure_premium)


In [ ]:
# GLM POISSON
from sklearn.linear_model import PoissonRegressor

glm_freq = PoissonRegressor(
    alpha=1e-4,
    max_iter=2000
)

glm_freq.fit(X_train_enc, y_train["FREQ"])

freq_test_pred = glm_freq.predict(X_test_enc)
freq_test_pred = np.clip(freq_test_pred, 1e-4, None)


In [ ]:
# Prédictions pure premium
pp_xgb_test = np.clip(xgb_final.predict(X_test_enc), 0, None)
pp_glm_test = np.clip(glm_final.predict(X_test_enc), 0, None)

#blend
w = 0.9  # poids XGB
pp_test_pred = w * pp_xgb_test + (1 - w) * pp_glm_test
pp_test_pred = np.clip(pp_test_pred, 0, None)

# Reconstitution CHARGE
annee_test = X_test["ANNEE_ASSURANCE"].values
charge_test_pred = pp_test_pred * annee_test

# CM 
cm_test_pred = charge_test_pred / (freq_test_pred * annee_test)
cm_test_pred = np.clip(cm_test_pred, 0, None)



In [ ]:
# Submission
submission4 = pd.DataFrame({
    "ID": X_test["ID"].values,
    "FREQ": freq_test_pred,
    "CM": cm_test_pred,
    "ANNEE_ASSURANCE": annee_test,
    "CHARGE": charge_test_pred
})


submission4.to_csv("submission_xgb_glm.csv", index=False)
print("submission_xgb_glm.csv généré")


In [ ]:
# Vérifications finales
print("Shape submission :", submission4.shape)
print("\nNaN par colonne :")
print(submission4.isna().sum())

print("\nStatistiques rapides :")
submission4.describe()


Le fichier de soumission contient exactement 95852 observations et les cinq variables attendues par la plateforme.
Aucune valeur manquante n’est présente, et toutes les contraintes métier élémentaires (positivité de la fréquence, du coût moyen et de la charge) sont respectées.

L’analyse descriptive montre que les distributions prédites sont cohérentes
avec celles observées sur les données d’apprentissage :
la fréquence moyenne reste faible, le coût moyen présente une forte dispersion
caractéristique de données à queue lourde, et la charge agrégée conserve
une asymétrie marquée sans générer de valeurs aberrantes numériquement instables.

Ces éléments confirment que l’amélioration de la performance RMSE observée
n’est pas due à un artefact numérique, mais résulte d’une meilleure
modélisation du signal sous-jacent.


In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
submission4_path = output_dir / "submission_xgb_glm.csv"
submission4.to_csv(submission4_path, index=False)
print("Soumission enregistrée")
print(submission4_path)
